# Defense baselines: AS fine-tuning vs UAT vs UAP fine-tuning vs PGD-AT

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/defense_baselines.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All defenses start from the same trained VGG-19, fine-tune for the same number of epochs with half of every batch
perturbed (‖·‖₂ ≤ 4), and are evaluated the same way:

| defense | perturbation in training |
|---|---|
| noise | random direction (control) |
| **AS fine-tuning** (ours) | AS universal perturbation, rebuilt on the current model every 60 steps |
| UAP fine-tuning | DeepFool-based UAP (ART), rebuilt every 60 steps |
| UAT | Universal Adversarial Training (Shafahi et al. 2020): one universal perturbation updated by a gradient step every batch |
| PGD-AT | per-image ℓ₂ PGD, 5 steps (Madry et al. 2018) |

Evaluation: clean accuracy; fooling rate of AS and UAP attacks rebuilt from scratch on each model (2 attack sets);
accuracy under per-image ℓ₂ PGD; wall-clock training time.

Use a T4 GPU. About 90 min. Results are saved after every defense, so a disconnect loses at most one.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy, json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline
from asnet.robust import (adv_finetune, uat_finetune, pgd_finetune, pgd_accuracy,
                          as_perturbation, uap_perturbation, noise_perturbation)

set_seed(0)
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(device)

## Settings

In [ ]:
SIZE = 64
EPOCHS = 20           # base model
FT_EPOCHS = 5         # every defense
DELTA = 4             # universal perturbation norm (training and test)
REFRESH = 60          # AS / UAP fine-tuning: rebuild v every REFRESH steps
PGD_EPS = 2.0         # per-image l2 budget for PGD-AT and the PGD check
N_ATTACK = 200
EVAL_SEEDS = [0, 1]
N_PGD_TEST = 1000     # test images used for the per-image PGD check
DEFENSES = ["noise", "AS fine-tuning", "UAP fine-tuning", "UAT", "PGD-AT"]

## Base model

In [ ]:
train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
train_loader = make_loader(train_aug, 128, shuffle=True)

base = build_model("vgg19", NUM_CLASSES, SIZE)
if os.path.exists("vgg19_bloodmnist.pt"):
    base.load_state_dict(torch.load("vgg19_bloodmnist.pt", map_location="cpu"))
    base.to(device)
else:
    base, _ = train_classifier(base, train_loader, make_loader(val, 256), epochs=EPOCHS, device=device)
    torch.save(base.state_dict(), "vgg19_bloodmnist.pt")
base.eval()

Xte, yte = stack_dataset(test, device=device)
Xa, ya = stack_dataset(subset(train, 100, seed=300), device=device)   # images for building training perturbations
v_shape = Xte[:1].shape
print("base test acc", (predict(base, Xte) == yte).float().mean().item())

## Evaluation function

In [ ]:
def evaluate_model(name, m, train_seconds):
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    rows = []
    clean = (predict(m, Xte) == yte).float().mean().item()
    pgd = pgd_accuracy(m, Xte[:N_PGD_TEST], yte[:N_PGD_TEST], PGD_EPS, steps=10)
    for seed in EVAL_SEEDS:
        Xs, ys = stack_dataset(subset(train, N_ATTACK, seed=500 + seed), device=device)
        v_as = as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed, log=lambda *a: None).v
        v_uap = uap_baseline(m, Xs, NUM_CLASSES, delta=DELTA, device=device)
        rows.append({"defense": name, "seed": seed, "clean acc": clean,
                     "AS adaptive": fooling_rate(m, Xte, v_as),
                     "UAP adaptive": fooling_rate(m, Xte, v_uap),
                     f"PGD-l2 acc (eps={PGD_EPS})": pgd,
                     "train time (s)": train_seconds})
    for p in m.parameters():
        p.requires_grad_(True)
    print(pd.DataFrame(rows).drop(columns="seed").mean(numeric_only=True).round(3).to_dict())
    return rows

results = []
def save():
    pd.DataFrame(results).to_csv("defense_baselines.csv", index=False)

In [ ]:
results += evaluate_model("none (base)", base, 0.0)
save()

## Fine-tune and evaluate each defense

Each cell trains one defense from the same base model and adds its row.

In [ ]:
set_seed(1)
m, t = adv_finetune(copy.deepcopy(base), train_loader, noise_perturbation(v_shape, DELTA),
                    epochs=FT_EPOCHS, refresh=REFRESH, device=device)
results += evaluate_model("noise", m, t); save()

In [ ]:
set_seed(1)
m, t = adv_finetune(copy.deepcopy(base), train_loader, as_perturbation(Xa, ya, DELTA),
                    epochs=FT_EPOCHS, refresh=REFRESH, device=device)
torch.save(m.state_dict(), "vgg19_as_ft.pt")
results += evaluate_model("AS fine-tuning", m, t); save()

In [ ]:
set_seed(1)
m, t = adv_finetune(copy.deepcopy(base), train_loader, uap_perturbation(Xa, NUM_CLASSES, DELTA),
                    epochs=FT_EPOCHS, refresh=REFRESH, device=device)
results += evaluate_model("UAP fine-tuning", m, t); save()

In [ ]:
set_seed(1)
m, t = uat_finetune(copy.deepcopy(base), train_loader, DELTA, epochs=FT_EPOCHS, device=device)
results += evaluate_model("UAT", m, t); save()

In [ ]:
set_seed(1)
m, t = pgd_finetune(copy.deepcopy(base), train_loader, PGD_EPS, epochs=FT_EPOCHS, device=device)
results += evaluate_model("PGD-AT", m, t); save()

## Results

In [ ]:
res = pd.DataFrame(results)
cols = ["clean acc", "AS adaptive", "UAP adaptive", f"PGD-l2 acc (eps={PGD_EPS})", "train time (s)"]
table = res.groupby("defense", sort=False)[cols].agg(["mean", "std"]).round(3)
table

In [ ]:
mean = res.groupby("defense", sort=False)[cols].mean()
std = res.groupby("defense", sort=False)[cols].std().fillna(0)
order = list(mean.index)
colors = ["#888", "#c9a227", "#3b6ea5", "#d9822b", "#8b5fbf", "#4a9b6e"][:len(order)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(4)
w = 0.8 / len(order)
for i, (d, c) in enumerate(zip(order, colors)):
    axes[0].bar(x + i * w - 0.4 + w / 2, mean.loc[d, cols[:4]] * 100, w, yerr=std.loc[d, cols[:4]] * 100,
                color=c, label=d, capsize=2)
axes[0].set_xticks(x, ["clean acc", "AS fooling", "UAP fooling", f"PGD acc"], fontsize=9)
axes[0].set(ylabel="%", title=f"robustness (universal delta = {DELTA}, PGD eps = {PGD_EPS})")
axes[0].legend(fontsize=7)

axes[1].barh(order, mean["train time (s)"] / 60, color=colors)
axes[1].set(xlabel="minutes on a T4", title=f"fine-tuning time ({FT_EPOCHS} epochs)")
axes[1].invert_yaxis()

for d, c in zip(order, colors):
    axes[2].scatter(mean.loc[d, "clean acc"] * 100, mean.loc[d, "AS adaptive"] * 100, s=80, color=c, label=d)
axes[2].set(xlabel="clean accuracy (%)", ylabel="AS adaptive fooling (%)  (lower = better)",
            title="accuracy vs robustness")
plt.tight_layout(); plt.savefig("defense_baselines.png", bbox_inches="tight"); plt.show()